# Data Loading & Exploratory Data Analysis

Loads all six Amazon review TSV files, validates the schema, and explores key statistics before preprocessing.

In [2]:
import os
import pandas as pd

DATA_DIR = "../data"

FILES = {
    "Apparel":   "Apparel_review.tsv",
    "Beauty":    "Beauty_review.tsv",
    "Books":     "Books_review.tsv",
    "Furniture": "Furniture_review.tsv",
    "Mobile":    "Mobile_review.tsv",
    "Outdoors":  "Outdoors_review.tsv",
}

EXPECTED_COLUMNS = [
    "marketplace", "customer_id", "review_id", "product_id", "product_parent",
    "product_title", "product_category", "star_rating", "helpful_votes",
    "total_votes", "vine", "verified_purchase", "review_headline",
    "review_body", "review_date",
]

## Load Files

In [4]:
dfs = {}

NUMERIC_COLS = {"star_rating": "Int64", "helpful_votes": "Int64", "total_votes": "Int64"}

for category, filename in FILES.items():
    df = pd.read_csv(
        os.path.join(DATA_DIR, filename),
        sep="\t",
        on_bad_lines="skip",
        engine="python",
        dtype=str,  # read everything as str first to avoid column-shift misparses
    )
    for col, cast in NUMERIC_COLS.items():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(cast)
    df["source_category"] = category
    dfs[category] = df
    print(f"{category:12s}: {df.shape[0]:>9,} rows, {df.shape[1]} columns")

Apparel     : 5,877,663 rows, 16 columns
Beauty      : 5,090,735 rows, 16 columns
Books       : 3,101,049 rows, 16 columns
Furniture   :   790,987 rows, 16 columns
Mobile      : 5,013,957 rows, 16 columns
Outdoors    : 2,298,620 rows, 16 columns


## Schema Validation

Each file should follow the same 15-column schema.

In [5]:
for category, df in dfs.items():
    actual  = set(df.columns) - {"source_category"}
    missing = set(EXPECTED_COLUMNS) - actual
    extra   = actual - set(EXPECTED_COLUMNS)
    status  = "OK" if not missing and not extra else "MISMATCH"
    print(f"{category:12s}: {status}  missing={missing or '-'}  extra={extra or '-'}")

Apparel     : OK  missing=-  extra=-
Beauty      : OK  missing=-  extra=-
Books       : OK  missing=-  extra=-
Furniture   : OK  missing=-  extra=-
Mobile      : OK  missing=-  extra=-
Outdoors    : OK  missing=-  extra=-


## Data Types & Null Counts

In [6]:
print(dfs["Apparel"][EXPECTED_COLUMNS].dtypes)

marketplace            str
customer_id            str
review_id              str
product_id             str
product_parent         str
product_title          str
product_category       str
star_rating          Int64
helpful_votes        Int64
total_votes          Int64
vine                   str
verified_purchase      str
review_headline        str
review_body            str
review_date            str
dtype: object


In [7]:
null_summary = pd.DataFrame(
    {cat: df[EXPECTED_COLUMNS].isnull().sum() for cat, df in dfs.items()}
)
null_summary["total"] = null_summary.sum(axis=1)
null_summary

,Apparel,Beauty,Books,Furniture,Mobile,Outdoors,total
marketplace,0,0,0,0,0,0,0
customer_id,0,0,0,0,0,0,0
review_id,0,0,0,0,0,0,0
product_id,0,0,0,0,0,0,0
product_parent,0,0,0,0,0,0,0
product_title,12,39,0,31,0,0,82
product_category,0,0,0,0,0,0,0
star_rating,0,1,0,0,0,0,1
helpful_votes,0,1,0,0,0,0,1
total_votes,0,1,0,0,0,0,1


## Star Rating Distribution

In [8]:
rating_dist = pd.DataFrame(
    {cat: df["star_rating"].dropna().astype(int).value_counts().sort_index()
     for cat, df in dfs.items()}
)
rating_dist.index.name = "star_rating"
rating_dist

,Apparel,Beauty,Books,Furniture,Mobile,Outdoors
star_rating,,,,,,
1,443159,454554,237814,73214,608566,161138
2,367791,262698,166122,43769,245225,109207
3,620487,396419,249587,73438,464879,178792
4,1141929,738061,585361,153468,1005755,416936
5,3304297,3239002,1862165,447098,2689532,1432547


## Review Date Range

In [9]:
date_range = {}
for cat, df in dfs.items():
    dates = pd.to_datetime(df["review_date"], errors="coerce")
    date_range[cat] = {"earliest": dates.min().date(), "latest": dates.max().date()}

pd.DataFrame(date_range).T

,earliest,latest
Apparel,2000-09-06,2015-08-31
Beauty,2000-10-31,2015-08-31
Books,1995-06-24,2005-10-14
Furniture,2000-03-17,2015-08-31
Mobile,2010-11-04,2015-08-31
Outdoors,1999-03-24,2015-08-31


## Helpfulness Vote Coverage

Proportion of reviews that have any community votes (`total_votes > 0`).

In [10]:
vote_coverage = {
    cat: round((df["total_votes"] > 0).sum() / len(df) * 100, 2)
    for cat, df in dfs.items()
}
pd.Series(vote_coverage, name="% reviews with votes")

Apparel      29.85
Beauty       43.43
Books        89.82
Furniture    45.88
Mobile       33.51
Outdoors     42.03
Name: % reviews with votes, dtype: float64

## Vine & Verified Purchase Flags

In [11]:
flags = {
    cat: {
        "vine_Y_%":               round((df["vine"] == "Y").sum() / len(df) * 100, 2),
        "verified_purchase_Y_%": round((df["verified_purchase"] == "Y").sum() / len(df) * 100, 2),
    }
    for cat, df in dfs.items()
}
pd.DataFrame(flags).T

,vine_Y_%,verified_purchase_Y_%
Apparel,0.04,89.95
Beauty,0.65,82.69
Books,0.00,7.38
Furniture,0.35,90.67
Mobile,0.00,95.14
Outdoors,0.14,87.99
